<a href="https://colab.research.google.com/github/Miriam-P26/Default_tarjetas_de_credito/blob/main/notebooks/Modelado_Default_Tarjetas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5. Modelado

Objetivo
En esta etapa se utilizará el conjunto de desarrollo definido previamente para entrenar y evaluar distintos modelos de clasificación capaces de predecir el incumplimiento de pago de clientes de tarjetas de crédito.

Las reglas de preparación identificadas durante el análisis exploratorio serán reutilizadas de forma consistente para garantizar la reproducibilidad del proceso.

Entradas
desarrollo.csv: conjunto de desarrollo utilizado para entrenamiento y evaluación.
lote_produccion.csv: conjunto reservado para simulación de producción y monitoreo.
Salidas esperadas
Modelos entrenados.
Métricas de evaluación.
Selección del modelo final.
Pipeline reutilizable para MLflow y despliegue.

In [3]:
import pandas as pd
import numpy as np

5.1 Carga del conjunto de desarrollo

In [4]:
# Carga del conjunto de desarrollo
url = "https://raw.githubusercontent.com/Miriam-P26/Default_tarjetas_de_credito/main/data/desarrollo.csv"

df = pd.read_csv(url)

df.shape

(27000, 25)

5.2 Aplicación de reglas de preparación

Se reutilizan las reglas de preparación definidas durante el análisis exploratorio con el objetivo de garantizar consistencia entre la fase exploratoria y el entrenamiento de los modelos.

Las transformaciones implementadas se aplicarán nuevamente antes de la división entre entrenamiento y evaluación.

In [5]:
def limpiar(datos):
    datos = datos.copy()

    datos["educacion"] = datos["educacion"].replace({
        0: 4,
        5: 4,
        6: 4
    })

    datos["estado_civil"] = datos["estado_civil"].replace({
        0: 3
    })

    return datos

In [6]:
df = limpiar(df)

df.shape

(27000, 25)

5.3 Definición de predictores y variable objetivo

In [7]:
X = df.drop(columns=["incumple", "ID"])
y = df["incumple"]

print(X.shape)
print(y.shape)

(27000, 23)
(27000,)


5.4 Construcción del pipeline de preparación

Con el objetivo de garantizar que las transformaciones puedan reproducirse de forma consistente durante entrenamiento, validación y producción, se construirá un pipeline de preparación para las variables numéricas y categóricas.

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [9]:
categoricas = [
    "sexo",
    "educacion",
    "estado_civil"
]

numericas = [
    col for col in X.columns
    if col not in categoricas
]

In [10]:
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [11]:
cat_pipe = Pipeline([
("imputer", SimpleImputer(strategy="most_frequent")),
("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [12]:
transformaciones = ColumnTransformer([
    ("num", num_pipe, numericas),
    ("cat", cat_pipe, categoricas)
])

In [13]:
# Verificación del pipeline de preparación

X_transformado = transformaciones.fit_transform(X)

X_transformado.shape

(27000, 29)

Resultado de la preparación

Luego de aplicar las transformaciones definidas para las variables numéricas y categóricas, el conjunto de predictores pasó de 23 variables originales a 29 variables transformadas. Parece que funciona el pipeline.



5.5 División Train/Test

Una vez definidas las reglas de preparación y transformación de los datos, el conjunto de desarrollo se divide en conjuntos de entrenamiento y evaluación.

La evaluación se realizará exclusivamente sobre observaciones no utilizadas durante el entrenamiento.

El conjunto de producción reservado anteriormente permanecerá completamente aislado y no participará en esta etapa.

In [14]:
from sklearn.model_selection import train_test_split

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

In [16]:
print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (20250, 23)
Test: (6750, 23)


In [17]:
print("Train target")
print(
    round(
        y_train.value_counts(normalize=True) * 100,
        2
    )
)

print("\nTest target")
print(
    round(
        y_test.value_counts(normalize=True) * 100,
        2
    )
)

Train target
incumple
0    77.88
1    22.12
Name: proportion, dtype: float64

Test target
incumple
0    77.88
1    22.12
Name: proportion, dtype: float64


5.6 Aplicación del pipeline a Train y Test

In [18]:
X_train_transformado = transformaciones.fit_transform(X_train)

X_test_transformado = transformaciones.transform(X_test)

In [19]:
print("Train transformado:", X_train_transformado.shape)

print("Test transformado:", X_test_transformado.shape)

Train transformado: (20250, 29)
Test transformado: (6750, 29)
